In [2]:
from pathlib import Path
from math import radians, sin, cos, sqrt, atan2

import numpy as np
import pandas as pd

In [3]:


PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name.lower() in ["notebooks", "eda"]:
    PROJECT_ROOT = PROJECT_ROOT.parent

print("Project root:", PROJECT_ROOT)

Project root: C:\dishu\Cleartrace


> Importing required libraries and defining project paths

In [4]:
input_path = PROJECT_ROOT / "data" / "processed" / "delhi_aqi_90d.csv"

df = pd.read_csv(input_path)

df["timestamp_hour"] = pd.to_datetime(
    df["timestamp_hour"],
    errors="coerce"
)

df = (
    df.sort_values(["station_name", "timestamp_hour"])
    .reset_index(drop=True)
)

print("Shape:", df.shape)
print("Stations:", df["station_name"].nunique())
print("Timestamp range:", df["timestamp_hour"].min(), "to", df["timestamp_hour"].max())

print("\nDtypes to fix:")
print(df[["wind_direction_10m", "relative_humidity_2m"]].dtypes)

Shape: (82004, 44)
Stations: 38
Timestamp range: 2026-04-10 21:00:00 to 2026-07-09 18:00:00

Dtypes to fix:
wind_direction_10m      int64
relative_humidity_2m    int64
dtype: object


## Task 1 — Datatype Conversion

wind_direction_10m and relative_humidity_2m are stored as int64.
Converting to float64 for two reasons:
- Consistency with other meteorological columns.
- wind_direction_10m needs float64 for sin/cos calculations.

In [5]:
df['wind_direction_10m'] = df['wind_direction_10m'].astype(float)
df['relative_humidity_2m'] = df['relative_humidity_2m'].astype(float)

print(df[['wind_direction_10m', 'relative_humidity_2m']].dtypes)

wind_direction_10m      float64
relative_humidity_2m    float64
dtype: object


## Task 2 - To add reliability column: 

- If the valid AQI hours> 50 % of total hours,  we flag the station as reliable
- If not, we dont

In [6]:
reliability = df.groupby('station_name')['aqi_calculation_valid'].mean()

df['station_reliability'] = df['station_name'].map(reliability) > 0.5

print("Reliability per station:")
print(reliability.sort_values())

Reliability per station:
station_name
Chandni Chowk, Delhi - IITM                         0.077386
IHBAS, Dilshad Garden,New Delhi - CPCB              0.382298
NSIT Dwarka, Delhi - CPCB                           0.445783
New Moti Bagh, Delhi - MHUA                         0.539388
Alipur, Delhi - DPCC                                0.700185
Sonia Vihar, Delhi - DPCC                           0.715941
Anand Vihar, New Delhi - DPCC                       0.721501
Mandir Marg, New Delhi - DPCC                       0.741891
North Campus, DU, Delhi - IMD                       0.745134
Vivek Vihar, Delhi - DPCC                           0.745598
Burari Crossing, New Delhi - IMD                    0.753012
Dr. Karni Singh Shooting Range, Delhi - DPCC        0.753939
Najafgarh, Delhi - DPCC                             0.764597
Jahangirpuri, Delhi - DPCC                          0.765060
ITO, New Delhi - CPCB                               0.765524
Punjabi Bagh, Delhi - DPCC                     

## Findings:
> We conclude that the 3 stations-
- Chandni Chowk, Delhi - IITM
- IHBAS, Dilshad Garden,New Delhi - CPCB
- NSIT Dwarka, Delhi - CPCB
> are  marked as unreliable, which confirms our findings from missingno heatmap

## Task 3 — Missing Value Imputation

Cleaning order:

1. Interpolate only complete internal gaps of 6 hours or fewer.
2. Detect synchronized city-wide outage timestamps.
3. Remove city-wide outage timestamps.
4. Proxy-fill remaining station-level gaps using nearby reliable stations.
5. Preserve interpolation and proxy-imputation flags.

City-wide outages must be removed before proxy filling because nearby
stations cannot provide useful values when nearly the entire network is missing.

In [28]:
pollutants = ["pm25", "pm10", "no2", "co", "so2", "o3"]

df_clean = df.copy()


def interpolate_short_gaps(series, max_gap=6):
    missing = series.isna()

    gap_group = (missing != missing.shift()).cumsum()
    gap_length = missing.groupby(gap_group).transform("sum")

    interpolated = series.interpolate(
        method="linear",
        limit_area="inside"
    )

    eligible = missing & (gap_length <= max_gap)

    result = series.copy()
    result.loc[eligible] = interpolated.loc[eligible]

    return result


for col in pollutants:
    missing_before = df_clean[col].isna()

    df_clean[col] = (
        df_clean
        .groupby("station_name")[col]
        .transform(lambda s: interpolate_short_gaps(s, max_gap=6))
    )

    df_clean[f"{col}_interpolated"] = (
        missing_before & df_clean[col].notna()
    )

print("Missing values after short-gap interpolation:")
print(df_clean[pollutants].isna().sum())

print("\nInterpolated values:")
print(
    df_clean[
        [f"{col}_interpolated" for col in pollutants]
    ].sum()
)

print("\nTotal missing before:", df[pollutants].isna().sum().sum())
print("Total missing after:", df_clean[pollutants].isna().sum().sum())

Missing values after short-gap interpolation:
pm25     8599
pm10     9822
no2      8522
co       8843
so2     21942
o3       9867
dtype: int64

Interpolated values:
pm25_interpolated    10577
pm10_interpolated     9847
no2_interpolated     10562
co_interpolated      11374
so2_interpolated      9611
o3_interpolated      10432
dtype: int64

Total missing before: 129998
Total missing after: 67595


In [29]:
remaining = df_clean[pollutants].isnull().sum()
print("Still needs proxy fill (gaps > 6 hours):")
print(remaining)
print(f"\nTotal remaining: {remaining.sum()}")

Still needs proxy fill (gaps > 6 hours):
pm25     8599
pm10     9822
no2      8522
co       8843
so2     21942
o3       9867
dtype: int64

Total remaining: 67595


## Defining the distance formula for the latitude and longitude based stations

In [30]:
def haversine_km(lat1, lon1, lat2, lon2):
    radius = 6371.0

    lat1 = radians(lat1)
    lon1 = radians(lon1)
    lat2 = radians(lat2)
    lon2 = radians(lon2)

    dlat = lat2 - lat1
    dlon = lon2 - lon1

    a = (
        sin(dlat / 2) ** 2
        + cos(lat1) * cos(lat2) * sin(dlon / 2) ** 2
    )

    return radius * 2 * atan2(sqrt(a), sqrt(1 - a))



In [31]:
df_pre_proxy = df_clean.copy()

In [32]:
pollutants = ["pm25", "pm10", "no2", "co", "so2", "o3"]

citywide_missing_summary = []

for col in pollutants:
    temp = (
        df_pre_proxy
        .groupby("timestamp_hour")
        .agg(
            total_stations=("station_name", "nunique"),
            missing_stations=(col, lambda s: s.isna().sum())
        )
        .reset_index()
    )

    temp["missing_station_ratio"] = temp["missing_stations"] / temp["total_stations"]

    citywide_missing_summary.append(
        {
            "pollutant": col,
            "hours_50pct_missing": (temp["missing_station_ratio"] >= 0.50).sum(),
            "hours_75pct_missing": (temp["missing_station_ratio"] >= 0.75).sum(),
            "hours_90pct_missing": (temp["missing_station_ratio"] >= 0.90).sum(),
            "max_missing_ratio": round(temp["missing_station_ratio"].max(), 3),
            "mean_missing_ratio": round(temp["missing_station_ratio"].mean(), 3)
        }
    )

citywide_missing_summary = pd.DataFrame(citywide_missing_summary)

display(citywide_missing_summary)

,pollutant,hours_50pct_missing,hours_75pct_missing,hours_90pct_missing,max_missing_ratio,mean_missing_ratio
0,pm25,106,103,100,1.0,0.105
1,pm10,106,104,100,1.0,0.120
2,no2,112,104,100,1.0,0.104
3,co,112,103,100,1.0,0.108
4,so2,114,112,104,1.0,0.268
5,o3,106,104,100,1.0,0.120


- Most PM2.5, PM10, NO2, and CO missing values occur during synchronized city-wide missing periods, so nearest-station proxy imputation is not effective for these pollutants.

- SO2 and O3 contain additional station-specific missingness, so nearest-station proxy imputation can fill many of their gaps. However, these proxy-filled values should be flagged because they are estimated, not directly observed.

In [33]:
pollutants = ["pm25", "pm10", "no2", "co", "so2", "o3"]

outage_rows = []

for col in pollutants:
    temp = (
        df_pre_proxy
        .groupby("timestamp_hour")
        .agg(
            total_stations=("station_name", "nunique"),
            missing_stations=(col, lambda s: s.isna().sum())
        )
        .reset_index()
    )

    temp["missing_station_ratio"] = temp["missing_stations"] / temp["total_stations"]
    temp["pollutant"] = col

    outage_rows.append(temp[temp["missing_station_ratio"] >= 0.90])

outage_df = pd.concat(outage_rows, ignore_index=True)

display(
    outage_df
    .sort_values(["timestamp_hour", "pollutant"])
    .head(100)
)

,timestamp_hour,total_stations,missing_stations,missing_station_ratio,pollutant
300,2026-04-16 14:00:00,38,38,1.0,co
200,2026-04-16 14:00:00,38,38,1.0,no2
504,2026-04-16 14:00:00,38,38,1.0,o3
100,2026-04-16 14:00:00,38,38,1.0,pm10
0,2026-04-16 14:00:00,38,38,1.0,pm25
...,...,...,...,...,...
519,2026-05-05 17:00:00,38,38,1.0,o3
115,2026-05-05 17:00:00,38,38,1.0,pm10
15,2026-05-05 17:00:00,38,38,1.0,pm25
418,2026-05-05 17:00:00,38,38,1.0,so2


In [34]:
outage_summary_by_date = (
    outage_df
    .assign(date=outage_df["timestamp_hour"].dt.date)
    .groupby(["date", "pollutant"])
    .size()
    .reset_index(name="high_missing_hours")
)

display(outage_summary_by_date)

,date,pollutant,high_missing_hours
0,2026-04-16,co,8
1,2026-04-16,no2,8
2,2026-04-16,o3,8
3,2026-04-16,pm10,8
4,2026-04-16,pm25,8
5,2026-04-16,so2,9
6,2026-05-02,co,7
7,2026-05-02,no2,7
8,2026-05-02,o3,7
9,2026-05-02,pm10,7


## Findings

- Missingness is concentrated in specific timestamp blocks rather than being randomly distributed.
- During these blocks, most pollutants are unavailable across nearly all stations.
- These periods represent network-wide or provider-level data outages.
- Removing these timestamps is more defensible than attempting spatial proxy imputation.

In [35]:
pollutants = ["pm25", "pm10", "no2", "co", "so2", "o3"]

outage_flags = []

for col in pollutants:
    temp = (
        df_pre_proxy
        .groupby("timestamp_hour")
        .agg(
            total_stations=("station_name", "nunique"),
            missing_stations=(col, lambda s: s.isna().sum())
        )
        .reset_index()
    )

    temp["missing_station_ratio"] = temp["missing_stations"] / temp["total_stations"]
    temp["pollutant"] = col
    temp["is_citywide_missing"] = temp["missing_station_ratio"] >= 0.90

    outage_flags.append(temp)

outage_flags = pd.concat(outage_flags, ignore_index=True)

citywide_outage_hours = (
    outage_flags
    .groupby("timestamp_hour")
    .agg(
        pollutants_citywide_missing=("is_citywide_missing", "sum")
    )
    .reset_index()
)

citywide_outage_hours["is_citywide_outage_hour"] = (
    citywide_outage_hours["pollutants_citywide_missing"] >= 4
)

outage_timestamps = citywide_outage_hours.loc[
    citywide_outage_hours["is_citywide_outage_hour"],
    "timestamp_hour"
]

print("City-wide outage hours detected:", len(outage_timestamps))

display(
    citywide_outage_hours[
        citywide_outage_hours["is_citywide_outage_hour"]
    ].head(50)
)

City-wide outage hours detected: 100


,timestamp_hour,pollutants_citywide_missing,is_citywide_outage_hour
137,2026-04-16 14:00:00,6,True
138,2026-04-16 15:00:00,6,True
139,2026-04-16 16:00:00,6,True
140,2026-04-16 17:00:00,6,True
141,2026-04-16 18:00:00,6,True
142,2026-04-16 19:00:00,6,True
143,2026-04-16 20:00:00,6,True
144,2026-04-16 21:00:00,6,True
523,2026-05-02 16:00:00,6,True
524,2026-05-02 17:00:00,6,True


In [36]:
rows_before = len(df_pre_proxy)

df_clean = df_pre_proxy[


    
    ~df_pre_proxy["timestamp_hour"].isin(outage_timestamps)
].copy()

rows_after = len(df_clean)

print("Rows before dropping outage hours:", rows_before)
print("Rows after dropping outage hours:", rows_after)
print("Rows dropped:", rows_before - rows_after)

Rows before dropping outage hours: 82004
Rows after dropping outage hours: 78204
Rows dropped: 3800


## Findings:
City-wide outage analysis showed that most high-missingness periods were clustered on specific dates, especially 2026-06-23 to 2026-06-25. Since these missing periods affected nearly all stations and pollutants simultaneously, they were treated as provider/data-availability outages rather than station-level gaps. These timestamps were removed from the cleaned master dataset instead of being imputed.


In [37]:
print("Missing values after dropping city-wide outage hours:")
print(df_clean[pollutants].isnull().sum())

print("\nTotal remaining missing pollutant values:")
print(df_clean[pollutants].isnull().sum().sum())

print("\nMissing percentage:")
print((df_clean[pollutants].isnull().mean() * 100).round(2))

Missing values after dropping city-wide outage hours:
pm25     4799
pm10     6023
no2      4723
co       5044
so2     18143
o3       6068
dtype: int64

Total remaining missing pollutant values:
44800

Missing percentage:
pm25     6.14
pm10     7.70
no2      6.04
co       6.45
so2     23.20
o3       7.76
dtype: float64


In [38]:
MIN_VALID_RATIO = 0.80
MAX_PROXY_DISTANCE_KM = 15.0

df_proxy_source = df_clean.copy()

station_meta = (
    df_proxy_source
    .groupby("station_name", as_index=False)
    .agg(
        latitude=("latitude", "first"),
        longitude=("longitude", "first")
    )
    .dropna(subset=["latitude", "longitude"])
)

proxy_fill_logs = []

for col in pollutants:
    flag_col = f"{col}_proxy_imputed"
    df_clean[flag_col] = False

    station_quality = (
        df_proxy_source
        .groupby("station_name")[col]
        .agg(
            available_count=lambda s: s.notna().sum(),
            total_count="size"
        )
        .reset_index()
    )

    station_quality["valid_ratio"] = (
        station_quality["available_count"]
        / station_quality["total_count"]
    )

    reliable_stations = station_quality.loc[
        station_quality["valid_ratio"] >= MIN_VALID_RATIO,
        "station_name"
    ].tolist()

    pivot = df_proxy_source.pivot_table(
        index="timestamp_hour",
        columns="station_name",
        values=col,
        aggfunc="first"
    )

    for station in df_clean["station_name"].dropna().unique():
        remaining_idx = df_clean.index[
            (df_clean["station_name"] == station)
            & (df_clean[col].isna())
        ]

        if len(remaining_idx) == 0:
            continue

        target_meta = station_meta[
            station_meta["station_name"] == station
        ]

        if target_meta.empty:
            continue

        target_lat = target_meta["latitude"].iloc[0]
        target_lon = target_meta["longitude"].iloc[0]

        candidates = station_meta[
            station_meta["station_name"].isin(reliable_stations)
            & (station_meta["station_name"] != station)
        ].copy()

        candidates["distance_km"] = candidates.apply(
            lambda row: haversine_km(
                target_lat,
                target_lon,
                row["latitude"],
                row["longitude"]
            ),
            axis=1
        )

        candidates = candidates[
            candidates["distance_km"] <= MAX_PROXY_DISTANCE_KM
        ].sort_values("distance_km")

        for _, candidate in candidates.iterrows():
            if len(remaining_idx) == 0:
                break

            proxy_station = candidate["station_name"]

            if proxy_station not in pivot.columns:
                continue

            timestamps = df_clean.loc[
                remaining_idx,
                "timestamp_hour"
            ]

            proxy_values = (
                pivot[proxy_station]
                .reindex(timestamps)
                .to_numpy()
            )

            available = pd.notna(proxy_values)

            if not available.any():
                continue

            fill_idx = remaining_idx[available]
            fill_values = proxy_values[available]

            df_clean.loc[fill_idx, col] = fill_values
            df_clean.loc[fill_idx, flag_col] = True

            proxy_fill_logs.append(
                {
                    "pollutant": col,
                    "target_station": station,
                    "proxy_station": proxy_station,
                    "distance_km": round(candidate["distance_km"], 2),
                    "filled_count": int(available.sum())
                }
            )

            remaining_idx = remaining_idx[~available]

proxy_fill_report = pd.DataFrame(proxy_fill_logs)

print("Missing values after proxy filling:")
print(df_clean[pollutants].isna().sum())

print("\nProxy-imputed values:")
print(
    df_clean[
        [f"{col}_proxy_imputed" for col in pollutants]
    ].sum()
)

print("\nProxy fill summary:")
display(
    proxy_fill_report
    .groupby("pollutant", as_index=False)
    .agg(
        filled_count=("filled_count", "sum"),
        proxy_stations_used=("proxy_station", "nunique")
    )
)


Missing values after proxy filling:
pm25     7
pm10     7
no2     13
co      13
so2     91
o3       7
dtype: int64

Proxy-imputed values:
pm25_proxy_imputed     4792
pm10_proxy_imputed     6016
no2_proxy_imputed      4710
co_proxy_imputed       5031
so2_proxy_imputed     18052
o3_proxy_imputed       6061
dtype: int64

Proxy fill summary:


,pollutant,filled_count,proxy_stations_used
0,co,5031,32
1,no2,4710,31
2,o3,6061,30
3,pm10,6016,30
4,pm25,4792,31
5,so2,18052,28


## Proxy-Fill Interpretation

Proxy filling was applied only after city-wide outage timestamps were removed.

Each imputed value is traceable through a pollutant-specific
`*_proxy_imputed` flag. Newly proxy-filled values were not reused as
sources for other stations.

In [39]:
master_clean = df_clean.copy()

print("Master clean shape:", master_clean.shape)
print("Stations:", master_clean["station_name"].nunique())

print("\nRemaining pollutant missing values:")
print(master_clean[pollutants].isna().sum())

print("\nRemaining pollutant missing percentage:")
print(
    (master_clean[pollutants].isna().mean() * 100)
    .round(3)
)

print("\nInterpolated counts:")
print(
    master_clean[
        [f"{col}_interpolated" for col in pollutants]
    ].sum()
)

print("\nProxy-imputed counts:")
print(
    master_clean[
        [f"{col}_proxy_imputed" for col in pollutants]
    ].sum()
)

Master clean shape: (78204, 57)
Stations: 38

Remaining pollutant missing values:
pm25     7
pm10     7
no2     13
co      13
so2     91
o3       7
dtype: int64

Remaining pollutant missing percentage:
pm25    0.009
pm10    0.009
no2     0.017
co      0.017
so2     0.116
o3      0.009
dtype: float64

Interpolated counts:
pm25_interpolated    10577
pm10_interpolated     9847
no2_interpolated     10562
co_interpolated      11374
so2_interpolated      9611
o3_interpolated      10432
dtype: int64

Proxy-imputed counts:
pm25_proxy_imputed     4792
pm10_proxy_imputed     6016
no2_proxy_imputed      4710
co_proxy_imputed       5031
so2_proxy_imputed     18052
o3_proxy_imputed       6061
dtype: int64


In [40]:
output_dir = PROJECT_ROOT / "data" / "processed"
output_dir.mkdir(parents=True, exist_ok=True)

output_path = output_dir / "master_clean.csv"

master_clean.to_csv(output_path, index=False)

print("Saved:", output_path)

Saved: C:\dishu\Cleartrace\data\processed\master_clean.csv


In [47]:
master_path = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "master_clean.csv"
)

master_check = pd.read_csv(master_path)

master_check["timestamp_hour"] = pd.to_datetime(
    master_check["timestamp_hour"],
    errors="coerce"
)

print("Shape:", master_check.shape)
print("Stations:", master_check["station_name"].nunique())
print("Timestamp min:", master_check["timestamp_hour"].min())
print("Timestamp max:", master_check["timestamp_hour"].max())

print("\nDuplicate station-hour rows:")
print(
    master_check.duplicated(
        subset=["station_name", "timestamp_hour"]
    ).sum()
)

print("\nMissing pollutant values:")
print(master_check[pollutants].isna().sum())

print("\nInterpolation flags:")
print(
    master_check[
        [f"{col}_interpolated" for col in pollutants]
    ].sum()
)

print("\nProxy-imputation flags:")
print(
    master_check[
        [f"{col}_proxy_imputed" for col in pollutants]
    ].sum()
)

print("\nAQI validity:")
print(
    master_check["aqi_calculation_valid"]
    .value_counts(dropna=False)
)

Shape: (78204, 57)
Stations: 38
Timestamp min: 2026-04-10 21:00:00
Timestamp max: 2026-07-09 18:00:00

Duplicate station-hour rows:
0

Missing pollutant values:
pm25     7
pm10     7
no2     13
co      13
so2     91
o3       7
dtype: int64

Interpolation flags:
pm25_interpolated    10577
pm10_interpolated     9847
no2_interpolated     10562
co_interpolated      11374
so2_interpolated      9611
o3_interpolated      10432
dtype: int64

Proxy-imputation flags:
pm25_proxy_imputed     4792
pm10_proxy_imputed     6016
no2_proxy_imputed      4710
co_proxy_imputed       5031
so2_proxy_imputed     18052
o3_proxy_imputed       6061
dtype: int64

AQI validity:
aqi_calculation_valid
True     60828
False    17376
Name: count, dtype: int64


## AQI Consistency Note

Pollutant columns were cleaned after AQI and subindex columns had already
been calculated.

Therefore, the existing AQI, rolling-average, subindex, and dominant-pollutant
columns still represent the original observed-data AQI pipeline. They were
not recalculated from proxy-imputed pollutant values.

Proxy-imputed pollutant values will be used as predictor inputs. Existing
AQI values will remain observation-derived targets.

## Downstream v2 correction

This notebook produces the intermediate `master_clean.csv` used by the original v1 feature pipeline. Its interpolated and proxy-imputed pollutant values remain explicitly identified through provenance flags.

The final forecasting dataset does not treat these filled values as original station observations. Notebook 04 reconstructs each `<pollutant>_observed` column using the original `has_<pollutant>` flags and rebuilds temporal features using observations only.

Therefore, `master_clean.csv` and `features_core_v1.csv` are retained as intermediate audit artifacts, while `features_core_v2.csv` is the authoritative modelling dataset.